<a href="https://colab.research.google.com/github/arpita1505/trial/blob/main/torch_compile.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install git+https://github.com/facebookresearch/segment-anything.git

  Cloning https://github.com/facebookresearch/segment-anything.git to /tmp/pip-req-build-21ir7gdl
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/segment-anything.git /tmp/pip-req-build-21ir7gdl
  Resolved https://github.com/facebookresearch/segment-anything.git to commit dca509fe793f601edb92606367a655c15ac00fdf
  Preparing metadata (setup.py) ... done


In [4]:
import torch
print(torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0))

2.10.0+cu128
CUDA available: True
Device: Tesla T4


In [5]:
!wget https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth

--2026-05-21 03:02:03--  https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 13.249.182.81, 13.249.182.39, 13.249.182.62, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|13.249.182.81|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 375042383 (358M) [binary/octet-stream]
Saving to: ‘sam_vit_b_01ec64.pth.1’

sam_vit_b_01ec64.pt 100%[===================>] 357.67M   282MB/s    in 1.3s    

2026-05-21 03:02:05 (282 MB/s) - ‘sam_vit_b_01ec64.pth.1’ saved [375042383/375042383]



In [8]:
!tar -xf screw.tar.xz

In [9]:
!find screw -name "*.png" | head -10


screw/ground_truth/thread_top/014_mask.png
screw/ground_truth/thread_top/021_mask.png
screw/ground_truth/thread_top/012_mask.png
screw/ground_truth/thread_top/019_mask.png
screw/ground_truth/thread_top/000_mask.png
screw/ground_truth/thread_top/002_mask.png
screw/ground_truth/thread_top/008_mask.png
screw/ground_truth/thread_top/003_mask.png
screw/ground_truth/thread_top/013_mask.png
screw/ground_truth/thread_top/016_mask.png


In [10]:
!find screw/test -name "*.png" | head -10

screw/test/thread_top/018.png
screw/test/thread_top/005.png
screw/test/thread_top/016.png
screw/test/thread_top/001.png
screw/test/thread_top/000.png
screw/test/thread_top/002.png
screw/test/thread_top/022.png
screw/test/thread_top/004.png
screw/test/thread_top/014.png
screw/test/thread_top/021.png


In [12]:
import torch
import time
import numpy as np
from PIL import Image
from segment_anything import sam_model_registry, SamPredictor

CHECKPOINT = "sam_vit_b_01ec64.pth"
DEVICE = "cuda"
IMAGE_PATH = "screw/test/thread_top/000.png"

# ── Load image ───────────────────────────────────────────────
image = np.array(Image.open(IMAGE_PATH).convert("RGB"))
print(f"Image shape: {image.shape}")

# ── Baseline ─────────────────────────────────────────────────
sam_base = sam_model_registry["vit_b"](checkpoint=CHECKPOINT)
sam_base.eval().to(DEVICE)
predictor_base = SamPredictor(sam_base)

def benchmark(predictor, image, n_runs=10):
    # Warmup run
    predictor.set_image(image)
    point = np.array([[image.shape[1]//2, image.shape[0]//2]])
    label = np.array([1])
    predictor.predict(point_coords=point, point_labels=label)

    times = []
    for _ in range(n_runs):
        start = time.perf_counter()
        predictor.set_image(image)
        predictor.predict(point_coords=point, point_labels=label)
        end = time.perf_counter()
        times.append(end - start)

    return np.mean(times), np.std(times)

print("\nBenchmarking baseline...")
base_mean, base_std = benchmark(predictor_base, image)
print(f"Baseline: {base_mean:.4f}s ± {base_std:.4f}s")

# ── torch.compile ─────────────────────────────────────────────
sam_compiled = sam_model_registry["vit_b"](checkpoint=CHECKPOINT)
sam_compiled.eval().to(DEVICE)

sam_compiled.image_encoder = torch.compile(
    sam_compiled.image_encoder,
    mode="reduce-overhead"
)

predictor_compiled = SamPredictor(sam_compiled)

# First run includes compile time — log it separately
print("\nFirst run (includes compile overhead)...")
first_start = time.perf_counter()
predictor_compiled.set_image(image)
first_end = time.perf_counter()
print(f"First run time: {first_end - first_start:.4f}s")

# Warm runs
print("\nBenchmarking compiled model (warm runs)...")
compiled_mean, compiled_std = benchmark(predictor_compiled, image)
print(f"Compiled: {compiled_mean:.4f}s ± {compiled_std:.4f}s")

# ── Summary ───────────────────────────────────────────────────
speedup = base_mean / compiled_mean
print("\n========== RESULTS ==========")
print(f"Baseline:          {base_mean:.4f}s ± {base_std:.4f}s")
print(f"torch.compile:     {compiled_mean:.4f}s ± {compiled_std:.4f}s")
print(f"Compile overhead:  {first_end - first_start:.4f}s (one-time cost)")
print(f"Speedup:           {speedup:.2f}x")
print(f"Latency reduction: {((base_mean - compiled_mean)/base_mean)*100:.1f}%")

Image shape: (1024, 1024, 3)

Benchmarking baseline...
Baseline: 0.3880s ± 0.0029s

First run (includes compile overhead)...


W0521 03:18:02.529000 1435 torch/_inductor/utils.py:1679] [0/0] Not enough SMs to use max_autotune_gemm mode


First run time: 36.8166s

Benchmarking compiled model (warm runs)...
Compiled: 0.3586s ± 0.0025s

========== RESULTS ==========
Baseline:          0.3880s ± 0.0029s
torch.compile:     0.3586s ± 0.0025s
Compile overhead:  36.8166s (one-time cost)
Speedup:           1.08x
Latency reduction: 7.6%
